# OPDI Full Pipeline on OpenSky Network

Runs the complete OPDI pipeline (steps 00-08) against the OpenSky S3 environment.
All intermediate tables are stored as parquet on `s3a://eurocontrol/opdi/`.

**Scope:** 1 day of data (2025-08-01), non-distributed mode.

**Pipeline stages:**

| Step | Description |
|------|-------------|
| 00e | OpenSky aircraft database |
| 00d | OurAirports reference data |
| 00a | Airport H3 detection zones |
| 00b | Airport ground layouts (OSM) |
| 00c | Airspace boundaries (ANSP/FIR/country) |
| 01 | State vector ingestion (direct S3 read) |
| 02 | Track processing |
| 03 | Flight list generation |
| 04 | Flight events & measurements |
| 05 | Parquet export |
| 07 | Basic statistics |
| 08 | Advanced statistics |

## 0. Setup

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from datetime import date, datetime
from opdi.config import OPDIConfig
from opdi.utils.spark_helpers import get_spark
from opdi.utils.storage import StorageManager

TARGET_DATE = date(2025, 8, 1)
TARGET_MONTH = date(TARGET_DATE.year, TARGET_DATE.month, 1)

# 20 major European airports for step 00b.
# Set to None to process ALL airports in the European bounding box.
AIRPORT_FILTER = [
    "EGLL",  # London Heathrow
    "LFPG",  # Paris CDG
    "EHAM",  # Amsterdam Schiphol
    "EDDF",  # Frankfurt
    "LEMD",  # Madrid Barajas
    "LEBL",  # Barcelona
    "LIRF",  # Rome Fiumicino
    "EDDM",  # Munich
    "EGKK",  # London Gatwick
    "EBBR",  # Brussels
    "LSZH",  # Zurich
    "LOWW",  # Vienna
    "EKCH",  # Copenhagen
    "ENGM",  # Oslo Gardermoen
    "ESSA",  # Stockholm Arlanda
    "LPPT",  # Lisbon
    "EIDW",  # Dublin
    "EPWA",  # Warsaw Chopin
    "LTFM",  # Istanbul
    "LGAV",  # Athens
]

config = OPDIConfig.for_environment("opensky")
print(f"Target date : {TARGET_DATE}")
print(f"Airport filter: {len(AIRPORT_FILTER) if AIRPORT_FILTER else 'ALL'} airports")

## 1. Create Spark session

In [ ]:
spark = get_spark("opensky", app_name="OPDI Full Pipeline")
storage = StorageManager(spark, config)
spark

## 2. Step 00e — OpenSky aircraft database

In [ ]:
from opdi.ingestion.osn_aircraft_db import AircraftDatabaseIngestion

acdb = AircraftDatabaseIngestion(spark, config)
acdb.create_table_if_not_exists()
count = acdb.ingest(mode="overwrite")
print(f"Aircraft DB: {count:,} records")

## 3. Step 00d — OurAirports reference data

In [ ]:
from opdi.ingestion.ourairports import OurAirportsIngestion

oa = OurAirportsIngestion(spark, config)
oa.create_tables()
stats = oa.ingest_all()
stats

## 4. Step 00a — Airport H3 detection zones

In [ ]:
from opdi.reference.h3_airport_zones import AirportDetectionZoneGenerator

zone_gen = AirportDetectionZoneGenerator(spark, config)
zones = zone_gen.generate()
zone_gen.save_prepared_to_table(max_radius_nm=30)

# Verify
df_zones = storage.read_table("h3_airport_detection_zones")
print(f"Airport detection zones: {df_zones.count():,} hex rows")

## 5. Step 00b — Airport ground layouts (OSM → H3)

Processes the airports in `AIRPORT_FILTER`. Set `AIRPORT_FILTER = None` above to process all airports.

In [ ]:
from opdi.reference.h3_airport_layouts import AirportLayoutGenerator, hexagonify_airport, HEXAERO_SCHEMA
import traceback

layout_gen = AirportLayoutGenerator(spark, config)
layout_gen.create_table_if_not_exists()

airports_to_process = AIRPORT_FILTER or layout_gen.fetch_airport_list().ident.to_list()
print(f"Processing {len(airports_to_process)} airports...")

success, failed = [], []
for icao in airports_to_process:
    print(f"  {icao}...", end=" ")
    result = layout_gen.process_airport(icao)
    if result is not None:
        success.append(icao)
        print(f"OK ({len(result)} hexes)")
    else:
        failed.append(icao)
        print("FAILED")

print(f"\nDone: {len(success)} success, {len(failed)} failed")
if failed:
    print(f"Failed airports: {failed}")

## 6. Step 00c — Airspace boundaries (ANSP / FIR / country → H3)

In [ ]:
from opdi.reference.h3_airspaces import AirspaceH3Generator

airspace_gen = AirspaceH3Generator(spark, config)
airspace_gen.create_table_if_not_exists()
airspace_gen.process_all()

# Verify
df_airspaces = storage.read_table("opdi_h3_airspace_ref")
print(f"Airspace hexes: {df_airspaces.count():,} rows")

## 7. Step 01 — State vector ingestion (direct S3 read)

Reads hourly partitions directly from the OpenSky S3 bucket and writes to the `osn_statevectors_v2` table via StorageManager.

In [ ]:
from opdi.ingestion.osn_statevectors import StateVectorIngestion

sv_ingestion = StateVectorIngestion(spark, config)
sv_ingestion.create_table_if_not_exists()

end_date = date(TARGET_DATE.year, TARGET_DATE.month, TARGET_DATE.day + 1)
rows = sv_ingestion.ingest_from_s3(start_date=TARGET_DATE, end_date=end_date)
print(f"Ingested {rows:,} state vectors for {TARGET_DATE}")

## 8. Step 02 — Track processing

Transforms raw state vectors into flight tracks with unique IDs, H3 encoding, distance, and cleaned altitude.

In [ ]:
from opdi.pipeline.tracks import TrackProcessor

track_proc = TrackProcessor(spark, config)
track_proc.create_table_if_not_exists()
track_proc.process_month(TARGET_MONTH)

# Verify
df_tracks = storage.read_table("osn_tracks")
print(f"Tracks: {df_tracks.count():,} state vectors with track IDs")
print(f"Unique tracks: {df_tracks.select('track_id').distinct().count():,}")

## 9. Step 03 — Flight list generation

Detects departures/arrivals using H3 airport zones, classifies flights, and enriches with aircraft metadata.

In [ ]:
from opdi.pipeline.flights import FlightListProcessor

fl_proc = FlightListProcessor(spark, config)
fl_proc.create_table_if_not_exists()

# DAI: departures, arrivals, internal flights
fl_proc.process_dai(TARGET_MONTH)

# Overflights: tracks with no airport match
fl_proc.process_overflights(TARGET_MONTH)

# Verify
df_flights = storage.read_table("opdi_flight_list")
print(f"Flight list: {df_flights.count():,} flights")
df_flights.select("ADEP", "ADES", "ICAO24", "FLT_ID", "DOF").show(10)

## 10. Step 04 — Flight events & measurements

Extracts flight phase transitions, FL crossings, airport entry/exit events, and first/last seen events.

In [ ]:
from opdi.pipeline.events import FlightEventProcessor

event_proc = FlightEventProcessor(spark, config)
event_proc.create_tables_if_not_exist()
event_proc.process_month(TARGET_MONTH)

# Verify
df_events = storage.read_table("opdi_flight_events")
df_measures = storage.read_table("opdi_measurements")
print(f"Flight events: {df_events.count():,}")
print(f"Measurements:  {df_measures.count():,}")

# Event type breakdown
df_events.groupBy("type").count().orderBy("count", ascending=False).show(15)

## 11. Step 05 — Parquet export

In [ ]:
from opdi.output.parquet_exporter import ParquetExporter

export_dir = "data/OPDI/v002"
exporter = ParquetExporter(spark, config, output_dir=export_dir)
export_stats = exporter.export_all(
    start_date=TARGET_DATE,
    end_date=date(TARGET_DATE.year, TARGET_DATE.month + 1, 1),
)
export_stats

## 12. Step 07 — Basic statistics

In [ ]:
from opdi.monitoring.basic_stats import BasicStatsCollector

stats_collector = BasicStatsCollector(spark, config)
stats_collector.print_summary(tables=[
    "osn_statevectors_v2",
    "osn_tracks",
    "opdi_flight_list",
    "opdi_flight_events",
    "opdi_measurements",
])

## 13. Step 08 — Advanced statistics & data quality

In [ ]:
from opdi.monitoring.advanced_stats import AdvancedStatsCollector

adv_stats = AdvancedStatsCollector(spark, config)
df_quality = adv_stats.generate_quality_report(
    "osn_statevectors_v2",
    "daily_row_counts.csv",
    "daily_row_counts.html",
)
df_quality.head()

## 14. Cleanup

In [ ]:
spark.stop()
print("Pipeline complete.")